In [1]:
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.CRITICAL, force=True, stream=sys.stdout)
# logging.
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
# logging.getLogger('torch_numopt.numerical_optimizer').setLevel(logging.DEBUG)
# logging.getLogger("torch_numopt.algorithms").setLevel(logging.DEBUG)
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.GradientDescent(model.parameters())
# opt = torch_numopt.GradientDescentLipschitz(model.parameters())
# opt = torch_numopt.ConjugateGradient(model.parameters())
# opt = torch_numopt.LBFGS(model.parameters())
# opt = torch_numopt.AdaHessian(model.parameters())
# opt = torch_numopt.GaussNewton(model.parameters())
opt = torch_numopt.LevenbergMarquardt(model.parameters())
# opt = torch_numopt.Newton(model.parameters(), lr_init=1e-2)
# opt = torch_numopt.NewtonCG(model.parameters(), lr_init=1e-3)

# opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_method="backtrack")
# opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_method="interpolate")
# opt = torch_numopt.GradientDescentLS(model.parameters(), line_search_method="bisect")
# opt = torch_numopt.ConjugateGradientLS(model.parameters())
# opt = torch_numopt.LBFGSLS(model.parameters(), line_search_cond="wolfe", line_search_method="interpolate")
# opt = torch_numopt.AdaHessianLS(model.parameters())
# opt = torch_numopt.GaussNewtonLS(model.parameters())
# opt = torch_numopt.NewtonLS(model.parameters())
# opt = torch_numopt.NewtonCGLS(model.parameters())

# opt = torch_numopt.GradientDescentTR(model.parameters())
opt = torch_numopt.NewtonTR(model.parameters(), trust_region_method="exact")
# opt = torch_numopt.GaussNewtonTR(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

Using exact trust region solver. Expect heavy memory use or even OOM exceptions.
epoch:  0, loss: 0.07910629361867905
epoch:  1, loss: 0.0349198542535305
epoch:  2, loss: 0.03277501091361046
epoch:  3, loss: 0.032078564167022705
epoch:  4, loss: 0.032078564167022705
epoch:  5, loss: 0.029888473451137543
epoch:  6, loss: 0.029888473451137543
epoch:  7, loss: 0.027957061305642128
epoch:  8, loss: 0.025495728477835655
epoch:  9, loss: 0.019015628844499588
epoch:  10, loss: 0.009427610784769058
epoch:  11, loss: 0.00783334020525217
epoch:  12, loss: 0.007446500938385725
epoch:  13, loss: 0.0072938366793096066
epoch:  14, loss: 0.0072938366793096066
epoch:  15, loss: 0.007187810260802507
epoch:  16, loss: 0.007187810260802507
epoch:  17, loss: 0.007023608312010765
epoch:  18, loss: 0.006947317160665989
epoch:  19, loss: 0.006834815721958876
epoch:  20, loss: 0.006834815721958876
epoch:  21, loss: 0.00678828451782465
epoch:  22, loss: 0.006753379013389349
epoch:  23, loss: 0.0067303478717803

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9980098009109497
Test metrics:  R2 = 0.9953783750534058
